In [0]:
# 1) Leer la capa BRONZE (datos crudos)
from pyspark.sql import functions as F

SCHEMA = "credito_riesgo_lab"
TABLA_BRONZE = f"workspace.{SCHEMA}.clientes_bronze"
TABLA_SILVER = f"workspace.{SCHEMA}.clientes_silver"

bronze = spark.table(TABLA_BRONZE)
print(f"Bronze: {bronze.count()} filas")

In [0]:
# 2) SILVER = limpieza y calidad
# Reglas (cada filter elimina filas "malas" que llegaron del origen):
#   a) dropDuplicates: quita duplicados por cliente_id
#   b) isNotNull: quita filas sin ingresos (dato crítico para el score)
#   c) rangos válidos: edad 18-100, endeudamiento 0-1, ltv 0-1.2
#   d) columna derivada útil: antigüedad en años
silver = (
    bronze
    .dropDuplicates(["cliente_id"])
    .filter(F.col("ingresos").isNotNull())
    .filter((F.col("edad") >= 18) & (F.col("edad") <= 100))
    .filter(F.col("endeudamiento").between(0, 1))
    .filter(F.col("ltv").between(0, 1.2))
    .withColumn("antiguedad_anios", F.round(F.col("antiguedad_meses") / 12, 1))
)

# Escritura inicial: primer borrador, SIN optimizar (la mejora llega en la celda siguiente)
(
    silver.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER)
)

n_bronze = spark.table(TABLA_BRONZE).count()
n_silver = spark.table(TABLA_SILVER).count()
print(f"Silver: {n_silver} filas limpias (de {n_bronze} crudas → se eliminaron {n_bronze - n_silver})")

In [0]:
# 2b) 🚀 MEJORA: coalesce(4) — menos ficheros pequeños en disco
# ¿Cuántos ficheros dejó la escritura anterior? DESCRIBE DETAIL nos lo dice
ficheros_antes = spark.sql(f"DESCRIBE DETAIL {TABLA_SILVER}").select("numFiles").first()[0]
print(f"Sin coalesce  → {ficheros_antes} ficheros en disco")

# coalesce(4) fusiona las particiones en 4 trozos ANTES de escribir.
# Es barato (no mueve datos por la red, solo junta trozos) y evita los
# "ficheros pequeños", que hacen lentas las lecturas posteriores.
(
    silver.coalesce(4).write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER)
)

ficheros_despues = spark.sql(f"DESCRIBE DETAIL {TABLA_SILVER}").select("numFiles").first()[0]
print(f"Con coalesce(4) → {ficheros_despues} ficheros en disco 🎯")
print("Con millones de filas (un banco real), menos ficheros = queries y dashboards más rápidos.")

In [0]:
# 3) Verificación: la silver ya no tiene duplicados ni nulos en ingresos
from pyspark.sql import functions as F

silver_check = spark.table(TABLA_SILVER)
print(f"Duplicados por cliente_id: {silver_check.count() - silver_check.dropDuplicates(['cliente_id']).count()}")
print(f"Ingresos nulos: {silver_check.filter(F.col('ingresos').isNull()).count()}")

display(silver_check.limit(5))

# 02 · Capa SILVER — Limpieza y calidad

**Qué hace una capa silver en la industria:** convertir los datos crudos en datos **limpios y confiables**: sin duplicados, sin nulos críticos, con rangos válidos y tipos correctos. Es la capa que consumen los data scientists.

**🚀 Optimización integrada (como en producción):** primero escribimos SIN optimizar y medimos, y en la última celda mejoramos con `coalesce(4)` → menos ficheros pequeños en disco.

> Lee `clientes_bronze` y escribe `clientes_silver`.